## Imports & Configuration

In [1]:
import json
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

KB_PATH = Path('knowledge_base.json')

VALID_FAILURE_TYPES = {'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'No Failure'}
VALID_AGE_BINS      = {'Young', 'Mid', 'Old'}
REQUIRED_KEYS       = {
    'id', 'failure_type', 'age_bin', 'recommended_actions',
    'part_codes', 'urgency_level', 'repair_manual_id', 'estimated_downtime_hours'
}

print('Imports OK')
print(f'Valid failure types : {VALID_FAILURE_TYPES}')
print(f'Valid age bins      : {VALID_AGE_BINS}')

Imports OK
Valid failure types : {'PWF', 'TWF', 'No Failure', 'RNF', 'OSF', 'HDF'}
Valid age bins      : {'Old', 'Young', 'Mid'}


## Knowledge Base Construction (JSON)

In [2]:
KNOWLEDGE_BASE = [
    # ── TWF  (Tool Wear Failure) ──────────────────────────────────────────────
    {
        'id': 1,
        'failure_type': 'TWF',
        'age_bin': 'Young',
        'recommended_actions': [
            'Inspect cutting edge geometry under 40x magnification and record wear pattern',
            'Replace indexable insert with ISO grade P30 carbide tip (part #T1042)',
            'Recalibrate feed-rate controller to manufacturer spec ±0.01 mm/rev',
            'Log replacement in CMMS and schedule 500-cycle re-inspection'
        ],
        'part_codes': ['T1042', 'T1043', 'LUB-220'],
        'urgency_level': 2,
        'repair_manual_id': 'RM-TWF-001',
        'estimated_downtime_hours': 1.5
    },
    {
        'id': 2,
        'failure_type': 'TWF',
        'age_bin': 'Mid',
        'recommended_actions': [
            'Perform full tool-holder runout check; replace if TIR > 0.005 mm',
            'Swap worn carbide insert set (P20 grade) and verify torque to 8 Nm',
            'Flush coolant lines and refill with 8% concentration emulsion',
            'Re-run break-in cycle at 60% spindle load for 15 minutes'
        ],
        'part_codes': ['T2210', 'TH-890', 'CLT-8PCT'],
        'urgency_level': 2,
        'repair_manual_id': 'RM-TWF-002',
        'estimated_downtime_hours': 2.0
    },
    {
        'id': 3,
        'failure_type': 'TWF',
        'age_bin': 'Old',
        'recommended_actions': [
            'Complete spindle bearing inspection; replace angular-contact pair if Lh < 200 h',
            'Replace full insert cartridge assembly (part #TC-4401) and anti-vibration shank',
            'Audit NC program tool-life counters and reset after replacement',
            'Schedule root-cause analysis meeting within 48 h of unplanned stoppage'
        ],
        'part_codes': ['TC-4401', 'AVS-12', 'BRG-7206'],
        'urgency_level': 3,
        'repair_manual_id': 'RM-TWF-003',
        'estimated_downtime_hours': 4.0
    },

    # ── HDF  (Heat Dissipation Failure) ──────────────────────────────────────
    {
        'id': 4,
        'failure_type': 'HDF',
        'age_bin': 'Young',
        'recommended_actions': [
            'Verify coolant flow rate at nozzle: target 12 L/min ±0.5 L/min',
            'Clean heat-exchanger fins with compressed air and degreaser spray',
            'Check thermal interface paste on drive IGBT module; reapply if dry',
            'Confirm ambient temperature in enclosure < 40 °C; add forced ventilation if needed'
        ],
        'part_codes': ['HX-FIN-01', 'TIM-PAD-3W', 'NZL-12L'],
        'urgency_level': 2,
        'repair_manual_id': 'RM-HDF-001',
        'estimated_downtime_hours': 1.0
    },
    {
        'id': 5,
        'failure_type': 'HDF',
        'age_bin': 'Mid',
        'recommended_actions': [
            'Replace coolant pump impeller (part #CP-7700) and inspect seal integrity',
            'Descale heat-exchanger tubes with 5% citric acid solution; flush 3× with DI water',
            'Re-calibrate over-temperature thermocouple; replace if drift > 2 °C',
            'Update thermal alarm setpoint in PLC from 85 °C to 80 °C as precaution'
        ],
        'part_codes': ['CP-7700', 'TC-TYPE-K', 'SEAL-V220'],
        'urgency_level': 2,
        'repair_manual_id': 'RM-HDF-002',
        'estimated_downtime_hours': 3.0
    },
    {
        'id': 6,
        'failure_type': 'HDF',
        'age_bin': 'Old',
        'recommended_actions': [
            'Immediate machine stop; allow 30-min cool-down before any contact',
            'Replace full coolant circuit: pump, reservoir, and all hose assemblies',
            'Inspect spindle motor winding insulation resistance (target > 100 MΩ)',
            'Submit thermal imaging report to engineering for drive derating review'
        ],
        'part_codes': ['CP-7700', 'HOSE-KIT-HD', 'IR-CAMERA-LOG'],
        'urgency_level': 3,
        'repair_manual_id': 'RM-HDF-003',
        'estimated_downtime_hours': 6.0
    },

    # ── PWF  (Power Failure) ──────────────────────────────────────────────────
    {
        'id': 7,
        'failure_type': 'PWF',
        'age_bin': 'Young',
        'recommended_actions': [
            'Measure DC bus voltage under load; acceptable range 560–620 V DC',
            'Tighten all power terminal lugs to specified torque (12 Nm for M6 terminals)',
            'Replace AC line filter capacitors if ESR > 0.5 Ω at 100 Hz',
            'Verify UPS bypass switch position and test auto-transfer within 20 ms'
        ],
        'part_codes': ['CAP-450V-470U', 'LUG-M6', 'UPS-BPS-01'],
        'urgency_level': 2,
        'repair_manual_id': 'RM-PWF-001',
        'estimated_downtime_hours': 2.0
    },
    {
        'id': 8,
        'failure_type': 'PWF',
        'age_bin': 'Mid',
        'recommended_actions': [
            'Replace servo drive IGBT module (part #IGM-1200A) following lockout procedure',
            'Inspect and clean main contactor contacts; replace if pitting depth > 0.3 mm',
            'Test regenerative braking resistor value; replace if deviation > 5%',
            'Check power factor correction bank for blown fuses; replace 32 A type gG'
        ],
        'part_codes': ['IGM-1200A', 'CTR-LC1D', 'RBR-10K-50W'],
        'urgency_level': 3,
        'repair_manual_id': 'RM-PWF-002',
        'estimated_downtime_hours': 4.5
    },
    {
        'id': 9,
        'failure_type': 'PWF',
        'age_bin': 'Old',
        'recommended_actions': [
            'Initiate emergency shutdown sequence per SOP-ELEC-07; tag out breaker',
            'Replace entire servo amplifier rack (part #SAR-6AX) with certified unit',
            'Commission electrician to perform insulation resistance test on all motor cables',
            'Schedule full electrical safety audit before return-to-service'
        ],
        'part_codes': ['SAR-6AX', 'INSUL-KIT', 'BREAKER-63A'],
        'urgency_level': 3,
        'repair_manual_id': 'RM-PWF-003',
        'estimated_downtime_hours': 8.0
    },

    # ── OSF  (Overstrain Failure) ─────────────────────────────────────────────
    {
        'id': 10,
        'failure_type': 'OSF',
        'age_bin': 'Young',
        'recommended_actions': [
            'Reduce spindle load by 15%; re-profile NC roughing pass depth of cut',
            'Inspect drive belt tension; adjust to 180 N ±10 N using tension gauge',
            'Verify workpiece fixture clamping torque matches process sheet ±5%',
            'Run adaptive control cycle to auto-calibrate torque limit parameters'
        ],
        'part_codes': ['BELT-B2241', 'TEN-GAUGE-01', 'CLAMP-KIT-F'],
        'urgency_level': 1,
        'repair_manual_id': 'RM-OSF-001',
        'estimated_downtime_hours': 0.5
    },
    {
        'id': 11,
        'failure_type': 'OSF',
        'age_bin': 'Mid',
        'recommended_actions': [
            'Replace motor drive belt (part #BELT-B2241) and realign pulleys to < 0.1 mm TIR',
            'Inspect motor bearing preload; adjust angular-contact nut to specified axial play',
            'Check load distribution across all axes; re-balance if delta > 8%',
            'Verify overload relay trip setting matches nameplate current ×1.15'
        ],
        'part_codes': ['BELT-B2241', 'BRG-6205-2RS', 'OLR-A9E'],
        'urgency_level': 2,
        'repair_manual_id': 'RM-OSF-002',
        'estimated_downtime_hours': 2.5
    },
    {
        'id': 12,
        'failure_type': 'OSF',
        'age_bin': 'Old',
        'recommended_actions': [
            'Stop machine immediately; inspect gearbox for tooth fracture under borescope',
            'Replace motor bearing set (part #BRG-7206-AC) and re-grease with NLGI 2',
            'Perform vibration spectrum analysis; compare to baseline from commissioning',
            'Review overload event history in SCADA; escalate if > 3 events in 7 days'
        ],
        'part_codes': ['BRG-7206-AC', 'GBX-INSPECT-KIT', 'VIB-SENSOR-01'],
        'urgency_level': 3,
        'repair_manual_id': 'RM-OSF-003',
        'estimated_downtime_hours': 5.0
    },

    # ── RNF  (Random / No-Fault Failure) ─────────────────────────────────────
    {
        'id': 13,
        'failure_type': 'RNF',
        'age_bin': 'Young',
        'recommended_actions': [
            'Perform full diagnostic cycle: check all I/O cards with PLC online monitor',
            'Review error log for transient fault codes; clear and re-test under load',
            'Inspect all connector crimps and cable shields for intermittent continuity',
            'If fault recurs within 24 h, escalate to OEM remote diagnostics session'
        ],
        'part_codes': ['DIAG-USB-KEY', 'IO-CARD-DI32', 'CONN-M12-8P'],
        'urgency_level': 1,
        'repair_manual_id': 'RM-RNF-001',
        'estimated_downtime_hours': 1.0
    },
    {
        'id': 14,
        'failure_type': 'RNF',
        'age_bin': 'Mid',
        'recommended_actions': [
            'Capture and export last 1000-line NC program trace for fault reconstruction',
            'Replace suspect encoder cable assembly (part #ENC-CBL-15M) with shielded type',
            'Verify EtherCAT ring topology integrity; check for packet loss > 0.001%',
            'Apply firmware patch v2.4.1 to CNC controller if pending update exists'
        ],
        'part_codes': ['ENC-CBL-15M', 'ECAT-SWITCH-4P', 'FW-PATCH-241'],
        'urgency_level': 1,
        'repair_manual_id': 'RM-RNF-002',
        'estimated_downtime_hours': 2.0
    },
    {
        'id': 15,
        'failure_type': 'RNF',
        'age_bin': 'Old',
        'recommended_actions': [
            'Replace CNC controller battery (CR2032); backup SRAM parameters before swap',
            'Perform full machine geometry check: squareness, parallelism, backlash',
            'Inspect all relay bases and timer modules for contact oxidation; replace if present',
            'Schedule comprehensive preventive overhaul within next 30-day window'
        ],
        'part_codes': ['BATT-CR2032', 'RELAY-MY2N', 'GEOM-CHECK-KIT'],
        'urgency_level': 2,
        'repair_manual_id': 'RM-RNF-003',
        'estimated_downtime_hours': 3.5
    },
]

with open(KB_PATH, 'w') as f:
    json.dump(KNOWLEDGE_BASE, f, indent=2)

print(f'Knowledge base saved  : {KB_PATH}')
print(f'Total entries         : {len(KNOWLEDGE_BASE)}')

# Summary table
kb_df = pd.DataFrame(KNOWLEDGE_BASE)
print('\nEntries per failure_type x age_bin:')
print(kb_df.groupby(['failure_type', 'age_bin']).size().unstack(fill_value=0))

Knowledge base saved  : knowledge_base.json
Total entries         : 15

Entries per failure_type x age_bin:
age_bin       Mid  Old  Young
failure_type                 
HDF             1    1      1
OSF             1    1      1
PWF             1    1      1
RNF             1    1      1
TWF             1    1      1


## Data Validation

In [3]:
def validate_knowledge_base(records: list) -> None:
    """
    Validate all knowledge-base records before persistence.
    Raises ValueError on the first offending record.
    """
    seen_ids = set()
    for i, entry in enumerate(records):
        # ── Required keys ────────────────────────────────────────────────────
        missing = REQUIRED_KEYS - entry.keys()
        if missing:
            raise ValueError(f'Record {i}: missing keys {missing}')

        # ── Duplicate IDs ─────────────────────────────────────────────────────
        rec_id = entry['id']
        if rec_id in seen_ids:
            raise ValueError(f'Record {i}: duplicate id={rec_id}')
        seen_ids.add(rec_id)

        # ── urgency_level ∈ {1, 2, 3} ─────────────────────────────────────────
        if entry['urgency_level'] not in {1, 2, 3}:
            raise ValueError(
                f'Record id={rec_id}: urgency_level={entry["urgency_level"]} '
                f'must be 1, 2, or 3'
            )

        # ── age_bin ∈ {"Young", "Mid", "Old"} ─────────────────────────────────
        if entry['age_bin'] not in VALID_AGE_BINS:
            raise ValueError(
                f'Record id={rec_id}: age_bin="{entry["age_bin"]}" '
                f'must be one of {VALID_AGE_BINS}'
            )

        # ── recommended_actions length ≥ 3 ────────────────────────────────────
        if len(entry['recommended_actions']) < 3:
            raise ValueError(
                f'Record id={rec_id}: recommended_actions must have ≥ 3 items'
            )

        # ── JSON serializability ──────────────────────────────────────────────
        try:
            json.dumps(entry)
        except (TypeError, ValueError) as exc:
            raise ValueError(f'Record id={rec_id}: not JSON-serializable — {exc}')

    print(f'Validation passed   : {len(records)} records, {len(seen_ids)} unique IDs')


validate_knowledge_base(KNOWLEDGE_BASE)

Validation passed   : 15 records, 15 unique IDs


## Maintenance Recommender Class (Cosine Similarity)

In [4]:
class MaintenanceRecommender:
    """
    Content-based maintenance recommender.

    Encodes (failure_type, age_bin) with OneHotEncoder and ranks knowledge-base
    entries by cosine similarity to the query vector.
    Compatible with FastAPI /predict endpoint integration.
    """

    # ─────────────────────────────────────────────────────────────────────────
    def __init__(self, kb_path: str) -> None:
        path = Path(kb_path)
        if not path.exists():
            raise FileNotFoundError(
                f'Knowledge base not found: {path.resolve()}'
            )

        with open(path, 'r') as f:
            records = json.load(f)

        self.kb_df: pd.DataFrame = pd.DataFrame(records)

        # ── OneHotEncoder ────────────────────────────────────────────────────
        self.encoder = OneHotEncoder(
            sparse_output=False,
            handle_unknown='ignore'
        )

        # Fit on (failure_type, age_bin) columns from knowledge base
        self.encoder.fit(self.kb_df[['failure_type', 'age_bin']])

        # Pre-compute encoded matrix for every knowledge-base entry  (N x F)
        self._kb_matrix: np.ndarray = self.encoder.transform(
            self.kb_df[['failure_type', 'age_bin']]
        )

        print(f'[MaintenanceRecommender] loaded {len(self.kb_df)} entries')
        print(f'  Encoder categories : {self.encoder.categories_}')
        print(f'  KB matrix shape    : {self._kb_matrix.shape}')

    # ─────────────────────────────────────────────────────────────────────────
    def get_recommendations(
        self,
        failure_type: str,
        age_bin: str,
        rul: float,
        top_k: int = 3
    ) -> list:
        """
        Return top-k maintenance recommendations ranked by cosine similarity.

        Parameters
        ----------
        failure_type : str  — one of TWF, HDF, PWF, OSF, RNF, No Failure
        age_bin      : str  — one of Young, Mid, Old
        rul          : float — Remaining Useful Life (hours) from regression
        top_k        : int  — number of recommendations to return (default 3)

        Returns
        -------
        list of dict
        """
        # Step 1 ── No Failure fast-path ──────────────────────────────────────
        if failure_type == 'No Failure':
            return [{'message': 'Machine is healthy', 'rul': float(rul)}]

        # Step 2 ── Input validation ───────────────────────────────────────────
        valid_ft = set(self.kb_df['failure_type'].unique())
        valid_ab = set(self.kb_df['age_bin'].unique())

        if failure_type not in valid_ft:
            raise ValueError(
                f'Invalid failure_type "{failure_type}". '
                f'Must be one of {sorted(valid_ft)}'
            )
        if age_bin not in valid_ab:
            raise ValueError(
                f'Invalid age_bin "{age_bin}". '
                f'Must be one of {sorted(valid_ab)}'
            )

        # Step 3 ── Build 2-D query array ─────────────────────────────────────
        query_array = np.array([[failure_type, age_bin]])   # shape (1, 2)

        # Step 4 ── Transform with OneHotEncoder ──────────────────────────────
        query_encoded = self.encoder.transform(query_array)  # shape (1, F)

        # Step 5 ── Cosine similarity between query and every KB entry ─────────
        sim_matrix = cosine_similarity(query_encoded, self._kb_matrix)  # (1, N)

        # Step 6 ── Flatten to 1-D array ──────────────────────────────────────
        scores: np.ndarray = sim_matrix.flatten()  # shape (N,)

        # Step 7 ── Indices of top-k highest scores ────────────────────────────
        top_indices = np.argsort(scores)[::-1][:top_k]

        # Step 8 ── Build result dictionaries ─────────────────────────────────
        results = []
        for idx in top_indices:
            row = self.kb_df.iloc[int(idx)].to_dict()

            # ── Dynamic urgency override ──────────────────────────────────────
            urgency_note: str | None = None
            if rul < 10.0 and int(row['urgency_level']) < 3:
                row['urgency_level'] = 3
                urgency_note = 'Elevated due to critically low Remaining Useful Life.'

            rec = {
                'id'                      : int(row['id']),
                'failure_type'            : str(row['failure_type']),
                'age_bin'                 : str(row['age_bin']),
                'recommended_actions'     : list(row['recommended_actions']),
                'part_codes'              : list(row['part_codes']),
                'urgency_level'           : int(row['urgency_level']),
                'repair_manual_id'        : str(row['repair_manual_id']),
                'estimated_downtime_hours': float(row['estimated_downtime_hours']),
                'match_score'             : round(float(scores[idx]), 6),
                'rul'                     : float(rul),
            }
            if urgency_note is not None:
                rec['urgency_note'] = urgency_note

            results.append(rec)

        # ── Sort: urgency_level DESC, then match_score DESC ───────────────────
        results.sort(
            key=lambda r: (r['urgency_level'], r['match_score']),
            reverse=True
        )

        return results


print('MaintenanceRecommender class defined.')

MaintenanceRecommender class defined.


## Simulated Inference Pipeline

In [5]:
# ── Simulated model outputs ───────────────────────────────────────────────────
predicted_failure = 'OSF'
predicted_age_bin = 'Old'
predicted_rul     = 8.5          # critically low → urgency override will fire

# ── Instantiate recommender ───────────────────────────────────────────────────
recommender = MaintenanceRecommender(str(KB_PATH))

# ── Run recommendation ────────────────────────────────────────────────────────
results = recommender.get_recommendations(
    failure_type=predicted_failure,
    age_bin=predicted_age_bin,
    rul=predicted_rul
)

# ── Formatted output ──────────────────────────────────────────────────────────
print('=' * 68)
print(f'  MAINTENANCE RECOMMENDATIONS')
print(f'  Failure : {predicted_failure}   Age bin : {predicted_age_bin}'
      f'   RUL : {predicted_rul} h')
print('=' * 68)

for rank, rec in enumerate(results, start=1):
    urgency_label = {1: 'LOW', 2: 'MEDIUM', 3: 'HIGH'}[rec['urgency_level']]
    print(f'\n  Rank #{rank} ─── Match Score : {rec["match_score"]:.4f}')
    print(f'  Failure Type : {rec["failure_type"]}   '
          f'Age Bin : {rec["age_bin"]}')
    print(f'  Urgency      : {rec["urgency_level"]} ({urgency_label})', end='')
    if 'urgency_note' in rec:
        print(f'  ⚠  {rec["urgency_note"]}', end='')
    print()
    print(f'  RUL          : {rec["rul"]} h')
    print(f'  Repair Manual: {rec["repair_manual_id"]}')
    print(f'  Est. Downtime: {rec["estimated_downtime_hours"]} h')
    print(f'  Part Codes   : {", ".join(rec["part_codes"])}')
    print('  Recommended Actions:')
    for i, action in enumerate(rec['recommended_actions'], start=1):
        print(f'    {i}. {action}')
    print('  ' + '-' * 64)

print()

# ── Edge case 1: No Failure ───────────────────────────────────────────────────
print('Edge case — No Failure:')
healthy = recommender.get_recommendations('No Failure', 'Young', rul=150.0)
print(' ', healthy)

# ── Edge case 2: Invalid inputs ───────────────────────────────────────────────
print('\nEdge case — Invalid failure_type:')
try:
    recommender.get_recommendations('XYZ', 'Young', rul=50.0)
except ValueError as e:
    print(f'  ValueError caught: {e}')

print('\nEdge case — Invalid age_bin:')
try:
    recommender.get_recommendations('TWF', 'Baby', rul=50.0)
except ValueError as e:
    print(f'  ValueError caught: {e}')

# ── Edge case 3: High RUL (no urgency override) ───────────────────────────────
print('\nEdge case — High RUL (no urgency override):')
safe_results = recommender.get_recommendations('TWF', 'Young', rul=120.0)
for r in safe_results:
    print(f"  id={r['id']}  urgency={r['urgency_level']}  score={r['match_score']}  "
          f"urgency_note={r.get('urgency_note', 'None')}")

print('\n✓ Simulated inference pipeline complete.')

[MaintenanceRecommender] loaded 15 entries
  Encoder categories : [array(['HDF', 'OSF', 'PWF', 'RNF', 'TWF'], dtype=object), array(['Mid', 'Old', 'Young'], dtype=object)]
  KB matrix shape    : (15, 8)
  MAINTENANCE RECOMMENDATIONS
  Failure : OSF   Age bin : Old   RUL : 8.5 h

  Rank #1 ─── Match Score : 1.0000
  Failure Type : OSF   Age Bin : Old
  Urgency      : 3 (HIGH)
  RUL          : 8.5 h
  Repair Manual: RM-OSF-003
  Est. Downtime: 5.0 h
  Part Codes   : BRG-7206-AC, GBX-INSPECT-KIT, VIB-SENSOR-01
  Recommended Actions:
    1. Stop machine immediately; inspect gearbox for tooth fracture under borescope
    2. Replace motor bearing set (part #BRG-7206-AC) and re-grease with NLGI 2
    3. Perform vibration spectrum analysis; compare to baseline from commissioning
    4. Review overload event history in SCADA; escalate if > 3 events in 7 days
  ----------------------------------------------------------------

  Rank #2 ─── Match Score : 0.5000
  Failure Type : OSF   Age Bin : Mid